# ThermoProfiler Tutorial
*Predicting thermal rock properties (TC, SHC, TD) for sedimentary rocks from well logs*

This notebook demonstrates how to use **ThermoProfiler** to predict:
- Thermal Conductivity (**TC**)  
- Specific Heat Capacity (**SHC**)  
- Thermal Diffusivity (**TD**)  

The workflow is:
1. Load data into pandas dataframe 
2. Clean well log data.  
3. Choose a machine learning model (XGBoost, Random Forest, AdaBoost, or Linear).  
4. Run predictions using pre-trained models.  
5. Export results for further analysis or visualization.


### Install and Import

Make sure you have the correct versions of the required libraries installed:  

```bash
# MAKE TO USE CORRECT VERSIONS
pip install scikit-learn==1.2.1
pip install numpy==1.25.2
pip install pandas>=1.5.3
pip install joblib>=1.2.0
pip install lasio>=0.30.0


In [1]:
import pandas as pd
import numpy as np
from joblib import load
import sys
import os
sys.path.append(os.path.abspath("..")) 

sys.path.append(os.path.abspath("../thermoprofiler")) 

### Step 1 – Load Input Data

ThermoProfiler expects your well log data as a **tab-delimited TXT** or **CSV** file that can be read into a `pandas.DataFrame`.

Your file must contain the following well log columns (if available):
- `RHOB` → bulk density  
- `PHIN` → neutron porosity  
- `VSH` → shale volume (gamma-ray index)  
- `VP` → compressional wave velocity  

In addition, you should provide:
- `Depth` → depth column (required)  
- Metadata such as `Stratigraphy`, `Rock_type`, `Lithology` (optional, but `Rock_type` is used for selecting the correct model)  

Missing logs are handled automatically → the tool will fall back to the best available log combination.

Example:
```python
import pandas as pd

# Load either a TXT (tab-delimited) or CSV file
df = pd.read_csv("input_logs.txt", sep="\t")   # for TXT
# df = pd.read_csv("input_logs.csv")           # for CSV

df.head()


In [2]:
df = pd.read_csv(
    r"C:\Users\dergun\Documents\SHC-TC-TD-Prediction-using-petrophysical-well-logs\notebook\Example_well_input (1).txt",
    encoding="utf-16",   
    sep="\t", 
    header=0
)
# Drop first row (often metadata) and reset index
df = df.drop(0).reset_index(drop=True)
df


,Depth,Stratigraphy,Temp,GR,CAL,PHIN,RHOB,VSH,Lithology,Rock_type
0,0.00,Quaternary-Saale,-999.25,8.59859,-999,-999,-999,-999.25,S,3
1,0.25,Quaternary-Saale,-999.25,8.59859,-999,-999,-999,0.065429,S,3
2,0.50,Quaternary-Saale,-999.25,7.65106,-999,-999,-999,0.0482012,S,3
3,0.75,Quaternary-Saale,-999.25,7.70233,-999,-999,-999,0.0491333,S,3
4,1.00,Quaternary-Saale,-999.25,9.43716,-999,-999,-999,0.0806757,S,3
...,...,...,...,...,...,...,...,...,...,...
4396,1099.00,nodata,-999.25,-999.25,-999,-999,-999,-999.25,nodata,nodata
4397,1099.25,nodata,-999.25,-999.25,-999,-999,-999,-999.25,nodata,nodata
4398,1099.50,nodata,-999.25,-999.25,-999,-999,-999,-999.25,nodata,nodata
4399,1099.75,nodata,-999.25,-999.25,-999,-999,-999,-999.25,nodata,nodata


### Step 2 – Preprocess and Clean Logs

ThermoProfiler provides a helper function to **clean and standardize your input DataFrame** before making predictions.  
This step ensures that:
- Column names are normalized (`RHOB`, `PHIN`, `VSH`, `VP`)  
- Missing or invalid values are handled (`-999` or blanks → converted to `NaN`)  
- Rock type IDs are formatted correctly (`1=Evaporites, 2=Carbonates, 3=Clastics`)  

```python
from thermoprofiler.preprocessing import clean_log_dataframe

df = clean_log_dataframe(df)



In [3]:
from thermoprofiler.preprocessing import clean_log_dataframe
df = clean_log_dataframe(df)

### Step 3 – Run Predictions

Now we can apply ThermoProfiler to compute thermal rock properties from the logs.  
The core function is:

```python
from thermoprofiler.prediction import predict_all_properties

results = predict_all_properties(df, model_type="XGBOOST")


In [4]:

from thermoprofiler.prediction import predict_all_properties

df = predict_all_properties(df, model_type="LINEAR")

df.head()
df.value_counts()
df.tail(300)


,Depth,Stratigraphy,Temp,GR,CAL,PHIN,RHOB,VSH,Lithology,Rock_type,...,model_number_raw_SHC,model_number_TD,uncertainty_TD,model_number_raw_TD,TC_prediction,TC_prediction_raw,SHC_prediction,SHC_prediction_raw,TD_prediction,TD_prediction_raw
2845,711.25,Eocene/Paleocene,32.41,37.2469,265,0.2450,2.640,0.278086,Clst,3,...,11,8,6.4,11,2.831712,2.831712,1302.455299,1302.455299,1.102941,1.123854
2846,711.50,Eocene/Paleocene,32.41,37.2469,260,0.2955,2.578,0.278086,Clst,3,...,11,8,6.4,11,2.671861,2.671861,1462.896081,1462.896081,0.995475,1.017384
2847,711.75,Eocene/Paleocene,32.42,31.9933,260,0.3710,2.598,0.212416,Clst,3,...,11,8,6.4,11,2.589434,2.589434,1680.630152,1680.630152,0.864855,0.896313
2848,712.00,Eocene/Paleocene,32.4225,30.9173,262,0.3240,2.601,0.198966,Clst,3,...,11,8,6.4,11,2.726893,2.726893,1558.018145,1558.018145,0.971027,0.997861
2849,712.25,Eocene/Paleocene,32.43,35.6893,270,0.3510,2.598,0.258616,Clst,3,...,11,8,6.4,11,2.57312,2.57312,1610.67874,1610.67874,0.886277,0.915433
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3140,785.00,Postera,-999.25,59.8,47,NaN,2.628,NaN,Mrlst,3,...,1,1,19.6,1,2.832583,2.832583,831.958215,831.958215,1.302133,1.302133
3141,785.25,Postera,-999.25,58.7,47,NaN,2.544,NaN,Mrlst,3,...,1,1,19.6,1,2.658256,2.658256,991.597227,991.597227,1.196802,1.196802
3142,785.50,Postera,-999.25,53.2,47,NaN,2.462,NaN,Mrlst,3,...,1,1,19.6,1,2.488079,2.488079,1147.435309,1147.435309,1.093978,1.093978
3143,785.75,Postera,-999.25,51.8,47,NaN,2.480,NaN,Mrlst,3,...,1,1,19.6,1,2.525435,2.525435,1113.226949,1113.226949,1.116549,1.116549


### Step 4 – Export Results

After predictions are added to your DataFrame, you can export the results for further use.  
ThermoProfiler supports exporting to **TXT/CSV** (tab- or comma-delimited tables) and **LAS** (well log standard format).  

```python
from thermoprofiler.export import export_to_txt

# Save as tab-delimited TXT
export_to_txt(results, "predictions.txt")

# Save as CSV
results.to_csv("predictions.csv", index=False)



In [5]:
from thermoprofiler.export import export_to_txt, export_to_las
export_to_txt(df, "predictions.txt")

Predictions exported to TXT: c:\Users\dergun\Documents\thermoprofiler\notebook_tutorial\predictions.txt


In [6]:
export_to_las(df, "predictions.las", depth_col="Depth")

LAS file saved: c:\Users\dergun\Documents\thermoprofiler\notebook_tutorial\predictions.las
